In [0]:
from pyspark.sql.functions import col

In [0]:
# Lire la table sessions Bronze
df_sessions = spark.table("`E-commerce`.bronze.sessions")

In [0]:
# Afficher les types des colonnes
df_sessions.printSchema()

In [0]:
# Supprimer les lignes identiques
sessions_silver = df_sessions.dropDuplicates()

In [0]:
# Garder une seule ligne par session_id
sessions_silver = sessions_silver.dropDuplicates(["session_id"])

In [0]:
# Supprimer les sessions sans session_id

sessions_silver = sessions_silver.filter(
    col("session_id").isNotNull()
)

In [0]:
# Standardiser les valeurs de device
from pyspark.sql.functions import lower, trim

sessions_silver = sessions_silver.withColumn(
    "device",
    lower(trim(col("device")))
)

In [0]:
# Standardiser les valeurs de channel
sessions_silver = sessions_silver.withColumn(
    "channel",
    lower(trim(col("channel")))
)

In [0]:
# Remplacer les durees negatives par NULL
from pyspark.sql.functions import when

sessions_silver = sessions_silver.withColumn(
    "duration_seconds",
    when(col("duration_seconds") < 0, None)
    .otherwise(col("duration_seconds"))
)

In [0]:
# Remplacer les pages negatives par NULL
sessions_silver = sessions_silver.withColumn(
    "pages_viewed",
    when(col("pages_viewed") < 0, None)
    .otherwise(col("pages_viewed"))
)

In [0]:
# Lire les clients Silver
customers_clean = spark.table("`E-commerce`.silver.customers")

In [0]:
# Trouver les sessions avec un customer_id inexistant
invalid_sessions = sessions_silver.join(
    customers_clean.select("customer_id"),
    "customer_id",
    "left_anti"
)

display(invalid_sessions)

In [0]:
# Comparer le nombre de lignes
print("Bronze :", df_sessions.count())
print("Silver :", sessions_silver.count())

In [0]:
# Enregistrer sessions dans Silver
sessions_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.silver.sessions")